In [3]:
mpAPI = "oM8Dn5NdnFzwit5MyRA6LqEbqmfuM417"
from mp_api.client import MPRester
import json

# Querying Data

## Query with Materials Project IDs

In [4]:
with MPRester(mpAPI) as mpr:
    docs = mpr.materials.summary.search(
        material_ids = ["mp-149", "mp-13", "mp-22526"]
    )

Retrieving SummaryDoc documents: 100%|██████████| 3/3 [00:00<?, ?it/s]


In [5]:
example_doc = docs[0]
mpid = example_doc.material_id
formula = example_doc.formula_pretty
print(f"Material ID: {mpid}, Formula: {formula}")

Material ID: mp-149, Formula: Si


In [6]:
# list of available fields
list_of_available_fields = mpr.materials.summary.available_fields
print(list_of_available_fields)

['builder_meta', 'nsites', 'elements', 'nelements', 'composition', 'composition_reduced', 'formula_pretty', 'formula_anonymous', 'chemsys', 'volume', 'density', 'density_atomic', 'symmetry', 'material_id', 'deprecated', 'deprecation_reasons', 'last_updated', 'origins', 'warnings', 'structure', 'property_name', 'task_ids', 'uncorrected_energy_per_atom', 'energy_per_atom', 'formation_energy_per_atom', 'energy_above_hull', 'is_stable', 'equilibrium_reaction_energy_per_atom', 'decomposes_to', 'xas', 'grain_boundaries', 'band_gap', 'cbm', 'vbm', 'efermi', 'is_gap_direct', 'is_metal', 'es_source_calc_id', 'bandstructure', 'dos', 'dos_energy_up', 'dos_energy_down', 'is_magnetic', 'ordering', 'total_magnetization', 'total_magnetization_normalized_vol', 'total_magnetization_normalized_formula_units', 'num_magnetic_sites', 'num_unique_magnetic_sites', 'types_of_magnetic_species', 'bulk_modulus', 'shear_modulus', 'universal_anisotropy', 'homogeneous_poisson', 'e_total', 'e_ionic', 'e_electronic',

## Query with Property Filters

In [7]:
with MPRester(mpAPI) as mpr:
    docs = mpr.materials.summary.search(
        elements=["Si", "O"],
        band_gap = (0.5, 1.0)
    )

KeyboardInterrupt: 

In [ ]:
formulas = [doc.formula_pretty for doc in docs]
unique_formulas = list(set(formulas))
print("Total of Obtained Data: ", len(formulas))
print("Number of Unique Formulas: ", len(unique_formulas))

Total of Obtained Data:  443
Number of Unique Formulas:  382


In [ ]:
#Property filters with requested fields 
with MPRester(mpAPI) as mpr:
    docs = mpr.materials.summary.search(
        elements=["Si", "O"],
        band_gap = (0.5, 1.0),
        fields = ["material_id", "formula_pretty","band_gap", "volume"]
    )

Retrieving SummaryDoc documents: 100%|██████████| 443/443 [00:00<?, ?it/s]


In [ ]:
example_doc = docs[0]
mpid = example_doc.material_id
formula = example_doc.formula_pretty
bandgap = example_doc.band_gap 
print(f"Material ID: {mpid} with Formula: {formula} has the bandgap of {bandgap} eV")
#example_doc.fields_not_requested

Material ID: mp-34150 with Formula: SiO2 has the bandgap of 0.921899999999999 eV


## Filtering SiO2 Polymorhps

In [ ]:
sio2_docs = mpr.materials.summary.search(
    formula = "SiO2",
    fields = ["material_id", "formula_pretty", "band_gap",
              "energy_above_hull", "symmetry"]
)
#Sort by stability
sio2_sorted = sorted(sio2_docs, key=lambda x:x.energy_above_hull)
for doc in sio2_sorted[:5]:
    mpid = doc.material_id
    formula = doc.formula_pretty
    bandgap = doc.band_gap
    print(f"Material ID: {mpid}. Bandgap: {bandgap} eV. Energy above hull: {doc.energy_above_hull} eV/atom. Symmetry: {doc.symmetry}.")

Retrieving SummaryDoc documents: 100%|██████████| 321/321 [00:00<?, ?it/s]

Material ID: mp-7000. Bandgap: 5.719 eV. Energy above hull: 0.0 eV/atom. Symmetry: crystal_system=<CrystalSystem.trig: 'Trigonal'> symbol='P3_121' hall=None number=152 point_group='32' symprec=0.1 angle_tolerance=5.0 version='2.6.0'.
Material ID: mp-6930. Bandgap: 5.6695 eV. Energy above hull: 0.000625704444445 eV/atom. Symmetry: crystal_system=<CrystalSystem.trig: 'Trigonal'> symbol='P3_221' hall=None number=154 point_group='32' symprec=0.1 angle_tolerance=5.0 version='2.6.0'.
Material ID: mp-12787. Bandgap: 5.6273 eV. Energy above hull: 0.002772185555556 eV/atom. Symmetry: crystal_system=<CrystalSystem.mono: 'Monoclinic'> symbol='C2/c' hall=None number=15 point_group='2/m' symprec=0.1 angle_tolerance=5.0 version='2.6.0'.
Material ID: mp-7029. Bandgap: 5.5081 eV. Energy above hull: 0.00530613638889 eV/atom. Symmetry: crystal_system=<CrystalSystem.tet: 'Tetragonal'> symbol='P4_32_12' hall=None number=96 point_group='422' symprec=0.1 angle_tolerance=5.0 version='2.6.0'.
Material ID: mp-

## Filtering Perovskite

In [ ]:
#Query all entries from a certain formula
with MPRester(mpAPI) as mpr:
    docs = mpr.materials.summary.search(
        formula="ABC3",
        fields=["material_id", "formula_pretty", "band_gap"]
    )
docs[0]

## Determining Relaxation Functional

In [2]:
mp_id_to_task_id = {}
with MPRester(mpAPI) as mpr:
    summary_docs = mpr.materials.summary.search(
        material_ids = ["mp-149", "mp-13", "mp-22526"],
        fields = ["material_id", "structure", "origins", "formula_pretty"]
    )

    #Structure and origins are not included in the summary doc by default
    #Structure returns a pymatgen.Structure object to be converted to .cif
    #Origins returns a list of dictionaries associated with material's workflow

    for doc in summary_docs:
        for prop in doc.origins:
            if prop.name == "structure":
                mp_id_to_task_id[doc.material_id] = {
                    "task_id": prop.task_id,
                    "structure": doc.structure,
                    "formula": doc.formula_pretty
                }
                break
    
    thermo_docs = mpr.materials.thermo.search(
        material_ids = ["mp-149", "mp-13", "mp-22526"],
        fields = ["material_id", "entries"]
    )

for doc in thermo_docs:
    mp_id = doc.material_id
    for entry in doc.entries.values():
        if entry.data["task_id"] == mp_id_to_task_id[mp_id]["task_id"]:
            mp_id_to_task_id[mp_id]["run_type"] = entry.parameters["run_type"]
            break

Retrieving ThermoDoc documents: 100%|██████████| 9/9 [00:00<?, ?it/s]


In [11]:
mp_id_to_task_id[list(mp_id_to_task_id.keys())[0]]

{'task_id': MPID(mp-1947498),
 'structure': Structure Summary
 Lattice
     abc : 3.8492784033699095 3.8492794116013456 3.849278
  angles : 60.00001213094421 60.00000346645984 60.0000109754579
  volume : 40.32952684741405
       A : np.float64(3.333573) np.float64(0.0) np.float64(1.924639)
       B : np.float64(1.111191) np.float64(3.142924) np.float64(1.924639)
       C : np.float64(0.0) np.float64(0.0) np.float64(3.849278)
     pbc : True True True
 PeriodicSite: Si (3.889, 2.75, 6.736) [0.875, 0.875, 0.875]
 PeriodicSite: Si (0.5556, 0.3929, 0.9623) [0.125, 0.125, 0.125],
 'formula': 'Si',
 'run_type': 'r2SCAN'}

In [ ]:
mp_id_to_task_id[list(mp_id_to_task_id.keys())[2]]

{'task_id': MPID(mp-2206412),
 'structure': Structure Summary
 Lattice
     abc : 4.912370892891281 4.912370892891281 4.9123704771170225
  angles : 33.25407937825511 33.25407937825511 33.25406827000603
  volume : 31.733696867729037
       A : np.float64(4.70697107) np.float64(-1.4056282900000001) np.float64(-0.00449954)
       B : np.float64(4.70697107) np.float64(1.4056282900000001) np.float64(-0.00449954)
       C : np.float64(4.28950457) np.float64(0.0) np.float64(2.39406229)
     pbc : True True True
 PeriodicSite: Li (0.0, 0.0, 0.0) [-0.0, -0.0, -0.0]
 PeriodicSite: Co (6.852, 0.0, 1.193) [0.5, 0.5, 0.5]
 PeriodicSite: O (10.41, 0.0, 1.813) [0.76, 0.76, 0.76]
 PeriodicSite: O (3.289, 0.0, 0.5724) [0.24, 0.24, 0.24],
 'formula': 'LiCoO2',
 'run_type': 'r2SCAN'}

In [ ]:
#Query all entries from a certain chemical system with a specific functional 
chemsys = "Co-N"
with MPRester(mpAPI) as mpr:
    entries = mpr.get_entries_in_chemsys(
        chemsys,
        additional_criteria = {"thermo_types": [
            "GGA_GGA+U", "GGA_GGA+U_R2SCAN", "R2SCAN"]}
    )

entries[0]

## Other Data

In [25]:
with MPRester(mpAPI) as mpr:
    docs = mpr.materials.search(
        elements=["Si", "O"], num_sites=(0, 10),
        fields=["initial_structures"]
    )
                                              
example_doc = docs[0]
initial_structures = example_doc.initial_structures

Retrieving MaterialsDoc documents: 100%|██████████| 101/101 [00:00<?, ?it/s]


# Tips for Large Downloads

## Use `has_props` Key

Find which materials have desired properties instead of wasted queries

In [ ]:
with MPRester(mpAPI) as mpr:
    docs = mpr.materials.summary.search(
        has_props=["dielectric", "dos"], fields=["material_id"]
    )

## Restrict data to specific fields of interest using `fields`

In [ ]:
with MPRester(mpAPI) as mpr:
    docs = mpr.materials.summary.search(
        material_ids=["mp-149", "mp-13", "mp-22526"],
        fields=["material_id", "volume", "elements"]
    )

## Make a local copy to retrieve the data only once

In [29]:
with MPRester(mpAPI) as mpr:
    docs = mpr.materials.summary.search(
        material_ids = ["mp-149", "mp-13", "mp-22526"],
         fields = ["material_id", "formula_pretty", "band_gap"]
    )

#save to JSON
with open("materials_data.json", "w") as f:
    json.dump([doc.dict() for doc in docs], f, indent=4)

print("Data saved to materials_data.json")

Retrieving SummaryDoc documents: 100%|██████████| 3/3 [00:00<?, ?it/s]

Data saved to materials_data.json
